# چیلنج: ڈیٹا سائنس کے بارے میں متن کا تجزیہ

اس مثال میں، آئیے ایک سادہ مشق کرتے ہیں جو روایتی ڈیٹا سائنس کے عمل کے تمام مراحل کو شامل کرتی ہے۔ آپ کو کوئی کوڈ لکھنے کی ضرورت نہیں، آپ نیچے دیے گئے سیلز پر کلک کرکے انہیں چلا سکتے ہیں اور نتیجہ دیکھ سکتے ہیں۔ ایک چیلنج کے طور پر، آپ کو ترغیب دی جاتی ہے کہ اس کوڈ کو مختلف ڈیٹا کے ساتھ آزما کر دیکھیں۔

## مقصد

اس سبق میں، ہم نے ڈیٹا سائنس سے متعلق مختلف تصورات پر بات کی ہے۔ آئیے کچھ **متنی کان کنی** کرکے مزید متعلقہ تصورات دریافت کرنے کی کوشش کرتے ہیں۔ ہم ڈیٹا سائنس کے بارے میں ایک متن سے شروع کریں گے، اس سے کلیدی الفاظ نکالیں گے، اور پھر نتیجہ کو بصری شکل میں پیش کرنے کی کوشش کریں گے۔

بطور متن، میں ویکیپیڈیا کے صفحہ ڈیٹا سائنس کا استعمال کروں گا:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## مرحلہ 1: ڈیٹا حاصل کرنا

ہر ڈیٹا سائنس کے عمل کا پہلا مرحلہ ڈیٹا حاصل کرنا ہوتا ہے۔ ہم اس کے لیے `requests` لائبریری استعمال کریں گے:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## مرحلہ 2: ڈیٹا کو تبدیل کرنا

اگلا مرحلہ ڈیٹا کو ایسی شکل میں تبدیل کرنا ہے جو پراسیسنگ کے لیے موزوں ہو۔ ہمارے معاملے میں، ہم نے صفحے سے HTML سورس کوڈ ڈاؤن لوڈ کیا ہے، اور ہمیں اسے سادہ متن میں تبدیل کرنا ہے۔

اس کام کو کرنے کے بہت طریقے ہیں۔ ہم [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/) استعمال کریں گے، جو HTML پارسنگ کے لیے ایک معروف پائتھون لائبریری ہے۔ BeautifulSoup ہمیں مخصوص HTML عناصر پر توجہ مرکوز کرنے کی اجازت دیتا ہے، تاکہ ہم وکی پیڈیا کے مرکزی مضمون کے مواد پر توجہ مرکوز کر سکیں اور کچھ نیویگیشن مینو، سائیڈبار، فوٹرز، اور دیگر غیر متعلقہ مواد کو کم کر سکیں (حالانکہ کچھ بوائلر پلیٹ متن اب بھی رہ سکتا ہے)۔


سب سے پہلے، ہمیں HTML پارسنگ کے لیے BeautifulSoup لائبریری انسٹال کرنی ہوگی:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## مرحلہ 3: بصیرت حاصل کرنا

سب سے اہم قدم یہ ہے کہ ہمارے ڈیٹا کو ایسی شکل میں تبدیل کیا جائے جس سے ہم بصیرت حاصل کر سکیں۔ ہمارے معاملے میں، ہم متن سے اہم الفاظ نکالنا چاہتے ہیں، اور دیکھنا چاہتے ہیں کہ کون سے الفاظ زیادہ معنی خیز ہیں۔

ہم Python کی ایک لائبریری استعمال کریں گے جسے [RAKE](https://github.com/aneesha/RAKE) کہا جاتا ہے تاکہ اہم الفاظ نکالے جا سکیں۔ سب سے پہلے، اگر یہ لائبریری موجود نہیں تو اسے انسٹال کرتے ہیں: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

بنیادی فعالیت `Rake` آبجیکٹ سے دستیاب ہے، جسے ہم کچھ پیرامیٹرز کی مدد سے حسب ضرورت بنا سکتے ہیں۔ ہمارے معاملے میں، ہم ایک کی ورڈ کی کم از کم لمبائی 5 حروف، دستاویز میں کی ورڈ کی کم از کم تکرار 3، اور کی ورڈ میں الفاظ کی زیادہ سے زیادہ تعداد 2 مقرر کریں گے۔ آپ دوسرے اقدار کے ساتھ آزما کر نتیجہ دیکھ سکتے ہیں۔


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


ہم نے شرائط کی ایک فہرست حاصل کی جس کے ساتھ متعلقہ اہمیت کی مقدار بھی دی گئی ہے۔ جیسا کہ آپ دیکھ سکتے ہیں، سب سے متعلقہ شعبے، جیسے مشین لرننگ اور بگ ڈیٹا، فہرست میں اعلیٰ مقامات پر موجود ہیں۔

## مرحلہ 4: نتیجہ کی بصری نمائندگی

لوگ ڈیٹا کو بصری شکل میں بہتر انداز میں سمجھ سکتے ہیں۔ اس لیے بعض اوقات بصری شکل میں ڈیٹا کو دکھانا سمجھداری ہوتی ہے تاکہ کچھ بصیرتیں حاصل کی جا سکیں۔ ہم Python میں `matplotlib` لائبریری کا استعمال کر کے مطلوبہ الفاظ کی آسان تقسیم ان کی مطابقت کے ساتھ پلاٹ کر سکتے ہیں:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

تاہم، الفاظ کی تعدد کو دیکھانے کا ایک اور بھی بہتر طریقہ ہے - **ورڈ کلاؤڈ** کا استعمال۔ ہمیں اپنے کی ورڈ کی فہرست سے ورڈ کلاؤڈ بنانے کے لیے ایک اور لائبریری انسٹال کرنے کی ضرورت ہوگی۔


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` آبجیکٹ اصل متن، یا پہلے سے حساب شدہ الفاظ کی فہرست اور ان کی کثرت کو لیتا ہے، اور ایک تصویر واپس کرتا ہے، جسے پھر `matplotlib` کے ذریعے دکھایا جا سکتا ہے:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

ہم اصل متن کو بھی `WordCloud` میں دے سکتے ہیں - آئیے دیکھتے ہیں کہ کیا ہم قریب قریب نتیجہ حاصل کر سکتے ہیں:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

آپ دیکھ سکتے ہیں کہ ورڈ کلاؤڈ اب زیادہ متاثر کن لگتا ہے، لیکن اس میں بہت سا شور بھی شامل ہے (مثلاً غیر متعلقہ الفاظ جیسے `Retrieved on`)۔ نیز، ہمیں کم کلیدی الفاظ ملتے ہیں جو دو الفاظ پر مشتمل ہوتے ہیں، جیسے *data scientist*، یا *computer science*۔ اس کی وجہ یہ ہے کہ RAKE الگورتھم متن سے اچھے کلیدی الفاظ منتخب کرنے میں بہت بہتر کام کرتا ہے۔ یہ مثال ڈیٹا کی پیشگی پراسیسنگ اور صفائی کی اہمیت کو ظاہر کرتی ہے، کیونکہ آخر میں واضح تصویر ہمیں بہتر فیصلے کرنے کی اجازت دے گی۔

اس مشق میں ہم نے وکیپیڈیا کے متن سے کچھ معنی نکالنے کا ایک آسان عمل انجام دیا ہے، کلیدی الفاظ اور ورڈ کلاؤڈ کی صورت میں۔ یہ مثال کافی سادہ ہے، لیکن یہ تمام عام مراحل کو اچھی طرح دکھاتی ہے جو ایک ڈیٹا سائنسدان ڈیٹا کے ساتھ کام کرتے وقت اختیار کرتا ہے، ڈیٹا کے حصول سے لے کر بصری پیشکش تک۔

ہمارے کورس میں ہم ان تمام مراحل پر تفصیل سے گفتگو کریں گے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
